# Validate/Compare giFBA Results with Compartmentalized FBA

Following simulations are run in order:
- Synthetic Model/Scenarios (No monoculture)
- Real Models

## Synthetic Models


In [5]:
from cobra import Model
import pandas as pd
import numpy as np
import cobra as cb
import gifba
import optlang



for sit_idx in ["1c", "2a", "2b","2c", "2d","2e", "3a", "3b", "3c", "4a", "5a", "5b", "5c"]: 
    models, media = gifba.utils.load_simple_models(sit_idx)
    # rel abundances
    if sit_idx == "2b":
        rel_abund = [0.2, 0.8]
    elif sit_idx == "5b":
        rel_abund = [0.8, 0.2]
    elif sit_idx == "5c":
        rel_abund = [0.2, 0.8]
    else:
        rel_abund = [0.5, 0.5]

    print(f"\nSimulation {sit_idx} Models:")
    
    community = gifba.gifbaObject(models, media, rel_abund=rel_abund)
    print(community.objective_rxns)
    # comp_model, objective_reactions = gifba.utils.prepare_compartmentalized_model(community)
    comp_model, objective_reactions = prepare_compartmentalized_model(community)


    #========== Print compartmentalized model reactions =============
    for rxn in comp_model.reactions:
        # print original reaction
        print(rxn.id, ":  ", end="")
        for met in rxn.metabolites:
            if rxn.metabolites[met] < 0:
                print(rxn.metabolites[met], "*", met.id, end="  ")
        print(" --> ", end="")
        for met in rxn.metabolites:
            if rxn.metabolites[met] > 0:
                print(rxn.metabolites[met], "*", met.id, end="  ")

        print(" | LB:", rxn.lower_bound, " UB:", rxn.upper_bound)
        
    
    # reset all bounds
    for ex in comp_model.exchanges:
        ex.lower_bound = 0  # Set lower bound to 0 for all exchange reactions
        ex.upper_bound = 1000  # Set upper bound to 1000 for all exchange reactions
    
    # set media uptake bounds
    for ex, flux in media.items():
        ex = ex.upper() + "(e0)"
        comp_model.reactions.get_by_id(ex).lower_bound = flux  # set media uptake rates






    # ======== Optimize compartmentalized model for (optimal) community growth ========
    solution = comp_model.optimize()
    cb.io.save_json_model(comp_model, f"cFBA_Models/cFBA_sim{sit_idx}.json") # save for others to avoid re-building
    print(f"{comp_model.id}, Objective value: {solution.objective_value}")
    print(solution.fluxes)

    

    # ========= Set community growth constraint =============
    comm_growth = solution.fluxes[[rxn.id for rxn in objective_reactions]].sum()
    community_expr = sum(comp_model.reactions.get_by_id(f"EX_biomass{i+1}(e0)").flux_expression for i in range(len(models)))
    # Add constraint to model
    constraint = optlang.Constraint(
        community_expr,
        lb=comm_growth,  # lower bound
        ub=comm_growth,  # upper bound
        name="community_growth_constraint"
        )
    comp_model.solver.add(constraint)



    # ======= Get Min/Max Flux for organism 1 =============
    # minimize flux of first organism
    comp_model.objective = comp_model.reactions.get_by_id("EX_biomass1(e0)")
    comp_model.objective.direction = 'min'
    solution_min_s1 = comp_model.optimize()

    # maximize flux of first organism
    comp_model.objective.direction = 'max'
    solution_max_s1 = comp_model.optimize()

    # store results
    cfba_results = pd.DataFrame(columns=["First_Org_Flux", "Second_Org_Max_Flux", "Second_Org_Min_Flux"])

    

    # ======== Vary first organism flux between min and max and get second organism min/max ========
    for val in np.linspace(solution_min_s1.objective_value, solution_max_s1.objective_value, 101):
        # update constraint on first organism
        comp_model.reactions.get_by_id("EX_biomass1(e0)").upper_bound = val
        comp_model.reactions.get_by_id("EX_biomass1(e0)").lower_bound = val

        # optimize for second organism
        comp_model.objective = comp_model.reactions.get_by_id("EX_biomass2(e0)")
        comp_model.objective.direction = 'max'
        solution_max_s2 = comp_model.optimize()
        # minimize then maximize org 2
        comp_model.objective.direction = 'min'
        solution_min_s2 = comp_model.optimize()

        # store results
        cfba_results = pd.concat([cfba_results, pd.DataFrame({
            "First_Org_Flux": val,
            "Second_Org_Max_Flux": solution_max_s2.objective_value,
            "Second_Org_Min_Flux": solution_min_s2.objective_value,
        }, index=[0])], ignore_index=True)
        
    cfba_results.to_csv(f"Results/cFBA/cFBA_results_sim{sit_idx}.csv", index=False)






Simulation 1c Models:
Read LP format model from file /tmp/tmp0ndityia.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmp6zsja6tg.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros


{0: 'T_Bio', 1: 'T_Bio'}
Ex_A(e1) :  -2.0 * A[e1]   --> 1 * A[e0]   | LB: -2000.0  UB: 2000.0
T_A_m1 :  -1 * A[e1]   --> 1 * A[c1]   | LB: -2000.0  UB: 2000.0
T_Bio_m1 :  -1 * biomass1[c1]   --> 1 * biomass1[e1]   | LB: 0.0  UB: 2000.0
Biomass(c1) :  -1 * A[c1]   --> 1 * biomass1[c1]   | LB: 0.0  UB: 2000.0
EX_Bio(e1) :  -2.0 * biomass1[e1]   --> 1 * biomass1[e0]   | LB: -2000.0  UB: 2000.0
Ex_B(e2) :  -2.0 * B[e2]   --> 1 * B[e0]   | LB: -2000.0  UB: 2000.0
T_B_m2 :  -1 * B[e2]   --> 1 * B[c2]   | LB: -2000.0  UB: 2000.0
T_Bio_m2 :  -1 * biomass2[c2]   --> 1 * biomass2[e2]   | LB: 0.0  UB: 2000.0
Biomass(c2) :  -1 * B[c2]   --> 1 * biomass2[c2]   | LB: -2000.0  UB: 2000.0
EX_Bio(e2) :  -2.0 * biomass2[e2]   --> 1 * biomass2[e0]   | LB: -2000.0  UB: 2000.0
EX_A(e0) :  -1 * A[e0]   -->  | LB: -1000  UB: 1000
EX_biomass1(e0) :  -1 * biomass1[e0]   -->  | LB: -1000  UB: 1000
EX_B(e0) :  -1 * B[e0]   -->  | LB: -1000  UB: 1000
EX_biomass2(e0) :  -1 * biomass2[e0]   -->  | LB: -1000  UB: 10

/tmp/ipykernel_103144/1275788810.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({



Simulation 2a Models:
Read LP format model from file /tmp/tmp4t8kzbj9.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmp0z0n8mbi.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
{0: 'T_Bio', 1: 'T_Bio'}
Ex_A(e1) :  -2.0 * A[e1]   --> 1 * A[e0]   | LB: -2000.0  UB: 2000.0
T_A_m1 :  -1 * A[e1]   --> 1 * A[c1]   | LB: -2000.0  UB: 2000.0
T_Bio_m1 :  -1 * biomass1[c1]   --> 1 * biomass1[e1]   | LB: 0.0  UB: 2000.0
Biomass(c1) :  -1 * A[c1]   --> 1 * biomass1[c1]   | LB: 0.0  UB: 2000.0
EX_Bio(e1) :  -2.0 * biomass1[e1]   --> 1 * biomass1[e0]   | LB: -2000.0  UB: 2000.0
Ex_A(e2) :  -2.0 * A[e2]   --> 1 * A[e0]   | LB: -2000.0  UB: 2000.0
T_A_m2 :  -1 * A[e2]   --> 1 * A[c2]   | LB: -2000.0  UB: 2000.0
T_Bio_m2 :  -1 * biomass2[c2]   --> 1 * biomass2[e2]   | LB: 0.0  UB: 2000.0
Biomass(c2) :  -1 * A[c2]   --> 1 * biomass2[c2]   | LB: -2000.0  UB: 2000.0
EX_Bio(e2) :  -2.0 * biomass2[e2]   --> 1 * biomass2[e0]   | LB: -200

/tmp/ipykernel_103144/1275788810.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({


...

/tmp/ipykernel_103144/1275788810.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({



Simulation 5b Models:
Read LP format model from file /tmp/tmp3jwy91dj.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
Read LP format model from file /tmp/tmppmzu7vrg.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
{0: 'T_Bio', 1: 'T_Bio'}
Ex_A(e1) :  -1.25 * A[e1]   --> 1 * A[e0]   | LB: -1250.0  UB: 1250.0
Ex_B(e1) :  -1.25 * B[e1]   --> 1 * B[e0]   | LB: -1250.0  UB: 1250.0
T_A_m1 :  -1 * A[e1]   --> 1 * A[c1]   | LB: -1250.0  UB: 1250.0
T_B_m1 :  -1 * B[c1]   --> 1 * B[e1]   | LB: -1250.0  UB: 1250.0
T_Bio_m1 :  -1 * biomass1[c1]   --> 1 * biomass1[e1]   | LB: 0.0  UB: 1250.0
Biomass(c1) :  -1 * A[c1]   --> 1 * biomass1[c1]  1 * B[c1]   | LB: 0.0  UB: 1250.0
EX_Bio(e1) :  -1.25 * biomass1[e1]   --> 1 * biomass1[e0]   | LB: -1250.0  UB: 1250.0
Ex_B(e2) :  -5.0 * B[e2]   --> 1 * B[e0]   | LB: -5000.0  UB: 5000.0
Ex_A(e2) :  -5.0 * A[e2]   --> 1 * A[e0]   | LB: -5000.0  UB: 5000.0
T_B_m2 :  -1 * B[e2]   --> 1 * B[c2]   | LB: -5000.0  UB: 5000.0
T_A_m2

/tmp/ipykernel_103144/1275788810.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({



Simulation 5c Models:
Read LP format model from file /tmp/tmp6xdzg52p.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
Read LP format model from file /tmp/tmp5bq9n8tn.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
{0: 'T_Bio', 1: 'T_Bio'}
Ex_A(e1) :  -5.0 * A[e1]   --> 1 * A[e0]   | LB: -5000.0  UB: 5000.0
Ex_B(e1) :  -5.0 * B[e1]   --> 1 * B[e0]   | LB: -5000.0  UB: 5000.0
T_A_m1 :  -1 * A[e1]   --> 1 * A[c1]   | LB: -5000.0  UB: 5000.0
T_B_m1 :  -1 * B[c1]   --> 1 * B[e1]   | LB: -5000.0  UB: 5000.0
T_Bio_m1 :  -1 * biomass1[c1]   --> 1 * biomass1[e1]   | LB: 0.0  UB: 5000.0
Biomass(c1) :  -1 * A[c1]   --> 1 * biomass1[c1]  1 * B[c1]   | LB: 0.0  UB: 5000.0
EX_Bio(e1) :  -5.0 * biomass1[e1]   --> 1 * biomass1[e0]   | LB: -5000.0  UB: 5000.0
Ex_B(e2) :  -1.25 * B[e2]   --> 1 * B[e0]   | LB: -1250.0  UB: 1250.0
Ex_A(e2) :  -1.25 * A[e2]   --> 1 * A[e0]   | LB: -1250.0  UB: 1250.0
T_B_m2 :  -1 * B[e2]   --> 1 * B[c2]   | LB: -1250.0  UB: 1250.0
T_A_m2 

/tmp/ipykernel_103144/1275788810.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({


## Real Life Models

### Cross-Feeding E. coli & B. theta

In [1]:
def prepare_compartmentalized_model(community, rel_abund=None):
    from cobra import Model
    import cobra as cb

    models = community.models
    media = community.media
    rel_abund = list(community.rel_abund.flatten()) if rel_abund is None else 0#check_rel_abund(rel_abund, community.size)
    community_id = community.id
    community.create_vars()

    for model_idx, model in enumerate(models):
        # these will be converted to the internal reactions between compartment e1 or e2 moving to e0
        for ex in model.exchanges:
            ex.lower_bound = -1000  # Set lower bound to 0 for all exchange reactions
        for med_ex in media.keys():
            if med_ex in model.reactions:
                model.reactions.get_by_id(med_ex).lower_bound = media[med_ex] / rel_abund[model_idx]


    comp_model = Model("compartmentalized_model_"+str(community_id))
    met_mapping = dict({})
    compartments = []
    #======= Change Compartments (each model uses e0 and c{i+1} compartments)============	
    for model_idx, model in enumerate(models):
        # Change compartments for each model so that models[i] uses e0 and c{i+1} compartments
        for met in model.metabolites:
            # store original compartment and id
            orig_comp = met.compartment
            if orig_comp not in compartments:
                compartments.append(orig_comp)
            orig_id = met.id

            # adjust new compartment and id
            id = met.id.replace(f"biomass", f"biomass{model_idx +1}").replace(f"[c]", f"[c{model_idx +1}]").replace(f"[e]", f"[e{model_idx +1}]")
            met.id = id
            met.compartment = f"{orig_comp}{model_idx +1}"
            # map original id to new id
            met_mapping[id] = orig_id

            # add metabolite to compartmentalized model
            comp_model.add_metabolites([met.copy()])
    
    
    # change reaction names
    for model_idx, model in enumerate(models):
        for rxn in model.reactions:
            orig_id = rxn.id
            # orig_mets = rxn.metabolites
            orig_ub = rxn.upper_bound
            orig_lb = rxn.lower_bound

            
            id = orig_id.replace("(e)", f"(e{model_idx+1})").replace("(c)", f"(c{model_idx+1})")
            if id == orig_id:
                id = orig_id + f"_m{model_idx+1}"

            new_rxn = rxn.copy()
            new_rxn.id = id
            new_rxn.upper_bound = orig_ub / rel_abund[model_idx]
            new_rxn.lower_bound = orig_lb / rel_abund[model_idx]
            comp_model.add_reactions([new_rxn])
            
    # add e0 exchange rxns
    for rxn in comp_model.reactions:
        if rxn.boundary:
            # copy original ex_met 
            met_orig = list(rxn.metabolites.keys())[0]
            met_e0 = met_orig.copy()

            # adjust to move met from e{i+1} to e0
            met_e0.compartment = "e0"
            met_e0.id = met_orig.id.replace(f"[{met_orig.compartment}]", "[e0]")
            model_num = int(met_orig.compartment[-1]) -1
            print(model_num)
            comp_model.add_metabolites([met_e0])
            rxn.add_metabolites({
                met_e0: 1,
                # adjust for relative abundance and subtract 1 to account for original metabolite
                met_orig.id: (-1/rel_abund[model_num] + 1) 
            })

    # change bounds for e0 exchange reactions and add media uptake reactions
    for met in comp_model.metabolites:
        if met.compartment == "e0":
            ex = cb.Reaction(f"EX_{met.id}")
            ex.id = ex.id.replace("[", "(").replace("]", ")")
            ex.name = f"Exchange for {met.id}"
            ex.lower_bound = -1000  # allow uptake
            ex.upper_bound = 1000   # allow secretion
            ex.add_metabolites({met: -1})
            comp_model.add_reactions([ex])
        

    # Set objective to weighted sum of individual model biomass reactions
    objective_reactions = []
    for ex in comp_model.exchanges:
        if "biomass(e" in ex.id and not(ex.id.endswith("(e0)")):
            objective_reactions.append(ex)
    objective_rxns_coef = [1 for _ in range(len(models))]
    comp_model.objective = dict(zip(objective_reactions, objective_rxns_coef))

    print("Objective reactions:", objective_reactions)
    print("Objective coefficients:", objective_rxns_coef)

    return comp_model, objective_reactions

In [2]:
from cobra import Model
import pandas as pd
import numpy as np
import cobra as cb
import gifba
import optlang

EC_path = "AGORA2_Models/Escherichia_coli_str_K_12_substr_MG1655.mat"
EC = cb.io.load_matlab_model(EC_path)

BT_path = "AGORA2_Models/Bacteroides_thetaiotaomicron_3731.mat"
BT = cb.io.load_matlab_model(BT_path)

#example glucose minimal media
min_med_ids_ex = ['EX_glc_D(e)','EX_so4(e)','EX_nh4(e)','EX_pi(e)','EX_cys_L(e)',
            'EX_mn2(e)','EX_cl(e)','EX_ca2(e)','EX_mg2(e)','EX_cu2(e)',
            'EX_cobalt2(e)','EX_fe2(e)','EX_fe3(e)','EX_zn2(e)','EX_k(e)']
# Define medium uptake flux bounds
min_med_fluxes_ex = [-10,-100,-100,-100,-100,
                    -100,-100,-100,-100,-100,-100,-100,-100,-100,-100]
media = dict(zip(min_med_ids_ex, min_med_fluxes_ex))

# initialize community
community = gifba.gifbaObject([EC, BT], media, rel_abund=[0.5, 0.5])

# for ex in community.models[0].exchanges:
#     ex.lower_bound = 0  # Set lower bound to 0 for all exchange reactions
#     ex.upper_bound = 1000  # Set upper bound to 1000 for all exchange reactions
# for ex in community.models[1].exchanges:
#     ex.lower_bound = 0  # Set lower bound to 0 for all exchange reactions
#     ex.upper_bound = 1000  # Set upper bound to 1000 for all exchange reactions

# for ex in media.keys():
#     if ex in community.models[0].exchanges:
#         community.models[0].exchanges.get_by_id(ex).lower_bound = media[ex]
#     if ex in community.models[1].exchanges:
#         community.models[1].exchanges.get_by_id(ex).lower_bound = media[ex]



# comp_model, objective_reactions = gifba.utils.prepare_compartmentalized_model(community)
comp_model, objective_reactions = prepare_compartmentalized_model(community)

print(objective_reactions)


# reset all bounds
for ex in comp_model.exchanges:
    if len(ex.metabolites.keys()) == 1 and list(ex.metabolites.keys())[0].compartment == "e0":
        ex.lower_bound = 0  # Set lower bound to 0 for all exchange reactions
        ex.upper_bound = 1000  # Set upper bound to 1000 for all exchange reactions

# set media uptake bounds
for ex, flux in media.items():
    ex = ex.replace("(e)", "(e0)")
    comp_model.reactions.get_by_id(ex).lower_bound = flux  # set media uptake rates

#========== Print compartmentalized model reactions =============
# for rxn in comp_model.reactions:
    # if "biomass" in rxn.id.lower() or "bio1" in rxn.id.lower():
    #     print(rxn.id, ":  ", end="")
    #     for met in rxn.metabolites:
    #         if rxn.metabolites[met] < 0:
    #             print(rxn.metabolites[met], "*", met.id, end="  ")
    #     print(" --> ", end="")
    #     for met in rxn.metabolites:
    #         if rxn.metabolites[met] > 0:
    #             print(rxn.metabolites[met], "*", met.id, end="  ")
    #     print(" | LB:", rxn.lower_bound, " UB:", rxn.upper_bound)

    #     print()
    # print original reaction
    # print(rxn.id)
    # print(rxn.id, ":  ", end="")
    # for met in rxn.metabolites:
    #     if rxn.metabolites[met] < 0:
    #         print(rxn.metabolites[met], "*", met.id, end="  ")
    # print(" --> ", end="")
    # for met in rxn.metabolites:
    #     if rxn.metabolites[met] > 0:
    #         print(rxn.metabolites[met], "*", met.id, end="  ")
    






# ======== Optimize compartmentalized model for (optimal) community growth ========
solution = comp_model.optimize()
# cb.io.save_json_model(comp_model, f"cFBA_Models/cFBA_EC_BT.json")
print("Objective Direction:", comp_model.objective.direction)
print(f"{comp_model.id}, Objective value: {solution.objective_value}")
print("Objective reactions:", comp_model.objective.expression)

print(solution.fluxes)


# pull pareto front of compartmentalized model
# ========= Set community growth constraint =============
# comm_growth = solution.fluxes[[rxn.id for rxn in objective_reactions]].sum()
# community_expr = sum(comp_model.reactions.get_by_id(f"EX_biomass{i+1}(e0)").flux_expression for i in range(2))
# # Add constraint to model
# constraint = optlang.Constraint(
#     community_expr,
#     lb=comm_growth,  # lower bound
#     ub=comm_growth,  # upper bound
#     name="community_growth_constraint"
#     )
# comp_model.solver.add(constraint)



# ======= Get Min/Max Flux for organism 1 =============
# minimize flux of first organism
# comp_model.objective = comp_model.reactions.get_by_id("EX_biomass1(e0)")
# comp_model.objective.direction = 'min'
# solution_min_s1 = comp_model.optimize()

# # maximize flux of first organism
# comp_model.objective.direction = 'max'
# solution_max_s1 = comp_model.optimize()

# # store results
# cfba_results = pd.DataFrame(columns=["First_Org_Flux", "Second_Org_Max_Flux", "Second_Org_Min_Flux"])



# # ======== Vary first organism flux between min and max and get second organism min/max ========
# for val in np.linspace(solution_min_s1.objective_value, solution_max_s1.objective_value, 101):
#     # update constraint on first organism
#     comp_model.reactions.get_by_id("EX_biomass1(e0)").upper_bound = val
#     comp_model.reactions.get_by_id("EX_biomass1(e0)").lower_bound = val

#     # optimize for second organism
#     comp_model.objective = comp_model.reactions.get_by_id("EX_biomass2(e0)")
#     comp_model.objective.direction = 'max'
#     solution_max_s2 = comp_model.optimize()
#     # minimize then maximize org 2
#     comp_model.objective.direction = 'min'
#     solution_min_s2 = comp_model.optimize()

#     # store results
#     cfba_results = pd.concat([cfba_results, pd.DataFrame({
#         "First_Org_Flux": val,
#         "Second_Org_Max_Flux": solution_max_s2.objective_value,
#         "Second_Org_Min_Flux": solution_min_s2.objective_value,
#     }, index=[0])], ignore_index=True)
    
# cfba_results.to_csv(f"Results/cFBA/cFBA_results_EC_BT.csv", index=False)

Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p


Read LP format model from file /tmp/tmpidnlh72g.lp
Reading time = 0.01 seconds
: 2630 rows, 6554 columns, 22274 nonzeros
Read LP format model from file /tmp/tmpsbh4vs20.lp
Reading time = 0.00 seconds
: 1318 rows, 3010 columns, 11452 nonzeros
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [3]:
print("Objective direction:", comp_model.objective.direction)
print("Objective expression:", comp_model.objective.expression)

obj_rxn_coeffs = cb.util.solver.linear_reaction_coefficients(comp_model)
print("\nObjective reaction(s):")
for rxn, coeff in obj_rxn_coeffs.items():
    if coeff != 0:
        print(f"{rxn.id} (coefficient={coeff})")

Objective direction: min
Objective expression: 1.0*EX_biomass2(e0) - 1.0*EX_biomass2(e0)_reverse_f9757

Objective reaction(s):
EX_biomass2(e0) (coefficient=1.0)


In [2]:
import pandas as pd
import cobra as cb
import gifba 
from micom import Community
import numpy as np
import optlang

micom_results = pd.DataFrame(columns=["simulation", "tradeoff", "Org1_growth", "Org2_growth"])

for sit_idx in ["1c","2a","2b", "2c", "2d", "2e", "3a", "3b", "3c", "4a", "5a", "5b", "5c"]: # "2a", "2b","3a", "3b", "3c"
    # get media from giFBA
    models , media = gifba.utils.load_simple_models(sit_idx)
    id2index = {model.id: idx for idx, model in enumerate(models)}
    media = {k.replace("(e)", "_m"): -v for k, v in media.items()}
    rxn_up_bounds = {model_idx: {rxn.id : rxn.upper_bound for rxn in models[model_idx].reactions} for model_idx in range(len(models))}
    rxn_low_bounds = {model_idx: {rxn.id : rxn.lower_bound for rxn in models[model_idx].reactions} for model_idx in range(len(models))}

    print(rxn_up_bounds)

    # make sure 2b (same model, different abundances) uses correct abundances
    # abund = [0.2, 0.8] if sit_idx == "2b" else [0.5, 0.5]
    # sim_idx = sit_idx if sit_idx != "2b" else "2a"
    if sit_idx == "2b":
        abund = [0.2, 0.8]
        sim_idx = "2a"
    elif sit_idx == "5b":
        abund = [0.2, 0.8]
        sim_idx = "5a"
    elif sit_idx == "5c":
        abund = [0.8, 0.2]
        sim_idx = "5a"
    else:
        abund = [0.5, 0.5]
        sim_idx = sit_idx
    

    # create community dataframe
    community = pd.DataFrame({
        "id": ["Org1", "Org2"],
        "file": [f"../package/gifba/Simple_Models/sim{sim_idx}_1.json", f"../package/gifba/Simple_Models/sim{sim_idx}_2.json"],
        "abundance": abund
    })

    # create micom community
    community = Community(community)

    comp_model = cb.Model(f"compartmentalized_model_{sit_idx}")

    tmp = cb.Model("tmp")
    tmp.add_reactions([r.copy() for r in community.reactions])  # use community.model

    for rxn in tmp.reactions:
        # new_stoich = {}
        # for met, coef in list(rxn.metabolites.items()):  # snapshot (met->coef)
        #     if met.compartment == "m" and len(rxn.metabolites) == 1:
        #         new_stoich[met] = -1.0  # or coef * something, up to you
        #     elif met.compartment == "m":
        #         new_stoich[met] = 1.0  # or coef * something, up to you
        #     else:
        #         # WARNING: your compartments are like 'c__Org1', 'e__Org2' etc
        #         # met.compartment[-1] works only if last char is '1'/'2'
        #         model_num = int(met.compartment[-1]) - 1
        #         new_stoich[met] = coef / abund[model_num]
        
        # # overwrite stoichiometry
        # rxn.add_metabolites(new_stoich, combine=False)
        if "_m" not in rxn.id and len(rxn.metabolites) != 1:
            model_num = int(rxn.id.split("__")[-1][-1]) - 1
            orig_id = rxn.id.replace("__Org"+str(model_num+1), "")

            if "EX_" not in orig_id:
                rxn.lower_bound = rxn_low_bounds[model_num][orig_id] #* abund[model_num]
                rxn.upper_bound = rxn_up_bounds[model_num][orig_id]# * abund[model_num]


    comp_model.add_reactions([r.copy() for r in tmp.reactions])

    
    # for reaction in comp_model.reactions:
    #     print(reaction.id, reaction.reaction, reaction.lower_bound, reaction.upper_bound)
    
    # change objective to community growth (weighted sum of biomass reactions)
    objective_reactions = [rxn for rxn in comp_model.reactions if "ex_bio(e)" in rxn.id.lower()]
    objective_rxns_coef = abund #[1 for _ in range(len(objective_reactions))]
    comp_model.objective = dict(zip(objective_reactions, objective_rxns_coef))
    comp_model.objective.direction = "max"

    for rxn in comp_model.reactions:
        if len(rxn.metabolites) == 1 and list(rxn.metabolites.keys())[0].compartment == "m" and "biomass" not in rxn.id.lower():
        # if rxn.boundary and "_m" in rxn.id:
            # name = rxn.id.split("_m")[0]
            rxn.lower_bound = 0
            rxn.upper_bound = 1000
    
    for rxn in comp_model.reactions:
        if rxn.id in media:
            rxn.lower_bound = -media[rxn.id]  # set media uptake rates


    for reaction in comp_model.reactions:
        print(reaction.id, reaction.reaction, reaction.lower_bound, reaction.upper_bound)

    solution = comp_model.optimize()
    cb.io.save_json_model(comp_model, f"cFBA_Models/cFBA_sim{sit_idx}.json") # save for others to avoid re-building
    print(f"{comp_model.id}, Objective value: {solution.objective_value}")
    print(solution.fluxes)

    

    # ========= Set community growth constraint =============
    comm_growth = solution.fluxes[[rxn.id for rxn in objective_reactions]]@np.array(objective_rxns_coef)
    community_expr = sum(abund[i] *comp_model.reactions.get_by_id(f"EX_Bio(e)__Org{i+1}").flux_expression for i in range(len(models)))
    # Add constraint to model
    constraint = optlang.Constraint(
        community_expr,
        lb=comm_growth,  # lower bound
        ub=comm_growth,  # upper bound
        name="community_growth_constraint"
        )
    comp_model.solver.add(constraint)



    # ======= Get Min/Max Flux for organism 1 =============
    # minimize flux of first organism
    comp_model.objective = comp_model.reactions.get_by_id("EX_Bio(e)__Org1")
    comp_model.objective.direction = 'min'
    solution_min_s1 = comp_model.optimize()

    # maximize flux of first organism
    comp_model.objective.direction = 'max'
    solution_max_s1 = comp_model.optimize()

    

    # store results
    cfba_results = pd.DataFrame(columns=["First_Org_Flux", "Second_Org_Max_Flux", "Second_Org_Min_Flux"])

    print("Min and Max flux for first organism:", abund[0] * solution_min_s1.objective_value, abund[0] * solution_max_s1.objective_value)
    

    # ======== Vary first organism flux between min and max and get second organism min/max ========
    for val in np.linspace(solution_min_s1.objective_value, solution_max_s1.objective_value, 101):
        # update constraint on first organism
        comp_model.reactions.get_by_id("EX_Bio(e)__Org1").upper_bound = val
        comp_model.reactions.get_by_id("EX_Bio(e)__Org1").lower_bound = val

        # optimize for second organism
        comp_model.objective = comp_model.reactions.get_by_id("EX_Bio(e)__Org2")
        comp_model.objective.direction = 'max'
        solution_max_s2 = comp_model.optimize()
        # minimize then maximize org 2
        comp_model.objective.direction = 'min'
        solution_min_s2 = comp_model.optimize()

        # store results
        cfba_results = pd.concat([cfba_results, pd.DataFrame({
            "First_Org_Flux": abund[0] * val,
            "Second_Org_Max_Flux": abund[1] * solution_max_s2.objective_value,
            "Second_Org_Min_Flux": abund[1] * solution_min_s2.objective_value,
        }, index=[0])], ignore_index=True)
        
    cfba_results.to_csv(f"Results/cFBA/cFBA_results_sim{sit_idx}.csv", index=False)




    

Output()

{0: {'EX_A(e)': 1000.0, 'T_A': 1000.0, 'T_Bio': 1000.0, 'Biomass(c)': 1000.0, 'EX_Bio(e)': 1000.0}, 1: {'EX_B(e)': 1000.0, 'T_B': 1000.0, 'T_Bio': 1000.0, 'Biomass(c)': 1000.0, 'EX_Bio(e)': 1000.0}}


Output()

EX_A(e)__Org1 A[e]__Org1 <=> 0.5 A_m -100 1000.0
T_A__Org1 A[e]__Org1 <=> A[c]__Org1 -1000.0 1000.0
T_Bio__Org1 biomass[c]__Org1 --> biomass[e]__Org1 0.0 1000.0
Biomass(c)__Org1 A[c]__Org1 --> biomass[c]__Org1 0.0 1000.0
EX_Bio(e)__Org1 biomass[e]__Org1 <=> 0.5 biomass_m -100 1000.0
EX_A_m A_m <=>  -10 1000
EX_biomass_m biomass_m <=>  -1000.0 1000.0
EX_B(e)__Org2 B[e]__Org2 <=> 0.5 B_m -100 1000.0
T_B__Org2 B[e]__Org2 <=> B[c]__Org2 -1000.0 1000.0
T_Bio__Org2 biomass[c]__Org2 --> biomass[e]__Org2 0.0 1000.0
Biomass(c)__Org2 B[c]__Org2 <=> biomass[c]__Org2 -1000.0 1000.0
EX_Bio(e)__Org2 biomass[e]__Org2 <=> 0.5 biomass_m -100 1000.0
EX_B_m B_m <=>  -10 1000
compartmentalized_model_1c, Objective value: 20.0
EX_A(e)__Org1      -20.0
T_A__Org1           20.0
T_Bio__Org1         20.0
Biomass(c)__Org1    20.0
EX_Bio(e)__Org1     20.0
EX_A_m             -10.0
EX_biomass_m        20.0
EX_B(e)__Org2      -20.0
T_B__Org2           20.0
T_Bio__Org2         20.0
Biomass(c)__Org2    20.0
EX_Bio(e)_

/tmp/ipykernel_126777/1532299996.py:157: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({


...

/tmp/ipykernel_126777/1532299996.py:157: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({


Output()

{0: {'EX_A(e)': 1000.0, 'EX_B(e)': 1000.0, 'T_A': 1000.0, 'T_B': 1000.0, 'T_Bio': 1000.0, 'Biomass(c)': 1000.0, 'EX_Bio(e)': 1000.0}, 1: {'EX_B(e)': 1000.0, 'EX_A(e)': 1000.0, 'T_B': 1000.0, 'T_A': 1000.0, 'Biomass(c)': 1000.0, 'T_Bio': 1000.0, 'EX_Bio(e)': 1000.0}}


EX_A(e)__Org1 A[e]__Org1 <=> 0.8 A_m -100 1000.0
EX_B(e)__Org1 B[e]__Org1 <=> 0.8 B_m -100 1000.0
T_A__Org1 A[e]__Org1 <=> A[c]__Org1 -1000.0 1000.0
T_B__Org1 B[c]__Org1 <=> B[e]__Org1 -1000.0 1000.0
T_Bio__Org1 biomass[c]__Org1 --> biomass[e]__Org1 0.0 1000.0
Biomass(c)__Org1 A[c]__Org1 --> B[c]__Org1 + biomass[c]__Org1 0.0 1000.0
EX_Bio(e)__Org1 biomass[e]__Org1 <=> 0.8 biomass_m -100 1000.0
EX_A_m A_m <=>  -10 1000
EX_B_m B_m -->  0 1000
EX_biomass_m biomass_m -->  0.0 1000.0
EX_B(e)__Org2 B[e]__Org2 <=> 0.2 B_m -100 1000.0
EX_A(e)__Org2 A[e]__Org2 <=> 0.2 A_m -100 1000.0
T_B__Org2 B[e]__Org2 <=> B[c]__Org2 -1000.0 1000.0
T_A__Org2 A[e]__Org2 <=> A[c]__Org2 -1000.0 1000.0
Biomass(c)__Org2 A[c]__Org2 + B[c]__Org2 --> biomass[c]__Org2 0.0 1000.0
T_Bio__Org2 biomass[c]__Org2 --> biomass[e]__Org2 0.0 1000.0
EX_Bio(e)__Org2 biomass[e]__Org2 <=> 0.2 biomass_m -100 1000.0
compartmentalized_model_5c, Objective value: 10.0
EX_A(e)__Org1       -6.25
EX_B(e)__Org1        6.25
T_A__Org1        

/tmp/ipykernel_126777/1532299996.py:157: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({


In [ ]:
import pandas as pd
import cobra as cb
import gifba 
from micom import Community
import numpy as np
import optlang

micom_results = pd.DataFrame(columns=["simulation", "tradeoff", "Org1_growth", "Org2_growth", ])

model_paths = [
        ["AGORA2_Models/Escherichia_coli_str_K_12_substr_MG1655.mat",
        "AGORA2_Models/Bacteroides_thetaiotaomicron_3731.mat"],
        ['AGORA2_Models/Bifidobacterium_longum_infantis_ATCC_15697.mat',
         'AGORA2_Models/Eubacterium_hallii_DSM_3353.mat'],
         ["AGORA2_Models/Clostridium_difficile_R20291.mat",
          "AGORA2_Models/Clostridium_hiranonis_TO_931_DSM_13275.mat",
          "AGORA2_Models/Bacteroides_thetaiotaomicron_3731.mat",
          "AGORA2_Models/Eggerthella_lenta_DSM_11767.mat"]
]
media_list = []

# EC/BT
#example glucose minimal media
min_med_ids_ex = ['EX_glc_D(e)','EX_so4(e)','EX_nh4(e)','EX_pi(e)','EX_cys_L(e)',
			'EX_mn2(e)','EX_cl(e)','EX_ca2(e)','EX_mg2(e)','EX_cu2(e)',
			'EX_cobalt2(e)','EX_fe2(e)','EX_fe3(e)','EX_zn2(e)','EX_k(e)']
# Define medium uptake flux bounds
min_med_fluxes_ex = [-10,-100,-100,-100,-100,
					-100,-100,-100,-100,-100,-100,-100,-100,-100,-100]
media = dict(zip(min_med_ids_ex, min_med_fluxes_ex))
media_list.append(media)

# BI/AH
media = {
	'EX_o2(e)': 0, #aerobic/anaerobic
	'EX_h2o(e)': -1000,
	'EX_pi(e)': -1000,
	'EX_fe2(e)': -1000,
	'EX_fe3(e)': -1000,
	'EX_zn2(e)': -1000,
	'EX_so4(e)': -1000,
	'EX_cu2(e)': -1000,
	'EX_k(e)': -1000,
	'EX_mg2(e)': -1000,
	'EX_mn2(e)': -1000,
	'EX_cd2(e)': -1000,
	'EX_cl(e)': -1000,
	'EX_ca2(e)': -1000,
	'EX_cobalt2(e)': -1000,
	'EX_glc_D(e)': -10,
	'EX_nh4(e)': -20,

	'EX_ribflv(e)': -1000,
	'EX_pnto_R(e)': -1000,
	'EX_nac(e)': -1000,
	'EX_his_L(e)': -1000,
	'EX_asn_L(e)': -1000,
	'EX_glycys(e)': -1000,

	'EX_lys_L(e)': -1000,
	'EX_ala_L(e)': -1000,
	'EX_met_L(e)': -1000,
	'EX_leu_L(e)': -1000,
	'EX_hxan(e)': -1000,    
    'EX_glyglu(e)': -1000
	# 'EX_ser_L(e)': -1 #-1
}
media_list.append(media)


# CDiff community
euro_media = "AGORA2_Models/euro_diet.tsv"
euro_media = pd.read_csv(euro_media, sep="\t", header=0, index_col=0).to_dict()
euro_media = euro_media["Flux Value"]
euro_media = {ex.replace("[e]", "(e)"): -flux for ex, flux in euro_media.items()}
euro_media["EX_cobalt2(e)"] = -1000 #cobalt
euro_media["EX_so4(e)"] = -1000 # sulfate
euro_media["EX_adn(e)"] = -1000 #Adenosine
euro_media["EX_cytd(e)"] = -1000 # cytidine
euro_media["EX_thymd(e)"] = -1000 # thymidine
euro_media["EX_hspg(e)"] = -1 # Heparan Sulfate Proteoglycan
euro_media["EX_26dap_M(e)"] = -10 # Meso-2,6-Diaminoheptanedioate
euro_media["EX_cgly(e)"] = -1000 # Cysteinylglycine - dipep
euro_media["EX_glyasn(e)"] = -1000 # Glycylasparagine - dipep
euro_media["EX_nmn(e)"] = -1000 # Nicotinamide ribotide
euro_media["EX_pheme(e)"] = -1000 #p heme
euro_media["EX_sheme(e)"] = -1000 # s heme
euro_media["EX_spmd(e)"] = -1000 # spermidine
media_list.append(euro_media)

names = ["EC_BT", "BI_AH", "Cdiff_Community"]



for sit_idx in range(3): # EC/BT, BI/AH, Cdiff Comm
    # get media from giFBA
    media = media_list[sit_idx]
    models = [cb.io.load_matlab_model(path) for path in model_paths[sit_idx]]
    if sit_idx == 2:
        community = gifba.gifbaObject(models, [media, 0.1], # minimal media
									rel_abund=[0.1, 0.3, 0.3, 0.3]) 
        media = community.media
        
    id2index = {model.id: idx for idx, model in enumerate(models)}
    media = {k.replace("(e)", "_m"): -v for k, v in media.items()}
    rxn_up_bounds = {model_idx: {rxn.id : rxn.upper_bound for rxn in models[model_idx].reactions} for model_idx in range(len(models))}
    rxn_low_bounds = {model_idx: {rxn.id : rxn.lower_bound for rxn in models[model_idx].reactions} for model_idx in range(len(models))}

    # print(rxn_up_bounds)

    # make sure 2b (same model, different abundances) uses correct abundances
    # abund = [0.2, 0.8] if sit_idx == "2b" else [0.5, 0.5]
    # sim_idx = sit_idx if sit_idx != "2b" else "2a"
    if sit_idx !=2:
        abund = [0.5, 0.5]
        ids = [f"Org{i+1}" for i in range(len(models))]
    else:
        abund = [0.1, 0.3, 0.3, 0.3]
        ids = [f"Org{i+1}" for i in range(len(models))]

    # create community dataframe
    community = pd.DataFrame({
        "id": ids,
        "file": model_paths[sit_idx],
        "abundance": abund
    })

    # create micom community
    community = Community(community)

    comp_model = cb.Model(f"compartmentalized_model_{sit_idx}")

    tmp = cb.Model("tmp")
    tmp.add_reactions([r.copy() for r in community.reactions])  # use community.model

    for rxn in tmp.reactions:
        new_stoich = {}
        for met, coef in list(rxn.metabolites.items()):  # snapshot (met->coef)
            if met.compartment == "m" and len(rxn.metabolites) == 1:
                new_stoich[met] = -1.0  # or coef * something, up to you
            elif met.compartment == "m":
                new_stoich[met] = 1.0  # or coef * something, up to you
            else:
                # WARNING: your compartments are like 'c__Org1', 'e__Org2' etc
                # met.compartment[-1] works only if last char is '1'/'2'
                model_num = int(met.compartment[-1]) - 1
                new_stoich[met] = coef / abund[model_num]
        
        # overwrite stoichiometry
        rxn.add_metabolites(new_stoich, combine=False)
        if "_m" not in rxn.id and len(rxn.metabolites) != 1:
            model_num = int(rxn.id.split("__")[-1][-1]) - 1
            orig_id = rxn.id.replace("__Org"+str(model_num+1), "")

            if "EX_" not in orig_id:
                rxn.lower_bound = rxn_low_bounds[model_num][orig_id] * abund[model_num]
                rxn.upper_bound = rxn_up_bounds[model_num][orig_id] * abund[model_num]


    comp_model.add_reactions([r.copy() for r in tmp.reactions])

    
    for reaction in comp_model.reactions:
        if "biomass" in reaction.id.lower() or "bio" in reaction.id.lower():
            print(reaction.id, reaction.reaction, reaction.lower_bound, reaction.upper_bound)
    
    # change objective to community growth (weighted sum of biomass reactions)
    objective_reactions = [rxn for rxn in comp_model.reactions if "dm_biomass(e)" in rxn.id.lower()]
    objective_rxns_coef = [1 for _ in range(len(objective_reactions))]
    comp_model.objective = dict(zip(objective_reactions, objective_rxns_coef))
    comp_model.objective.direction = "max"

    for rxn in comp_model.reactions:
        if len(rxn.metabolites) == 1 and list(rxn.metabolites.keys())[0].compartment == "m" and "biomass" not in rxn.id.lower():
        # if rxn.boundary and "_m" in rxn.id:
            # name = rxn.id.split("_m")[0]
            rxn.lower_bound = 0
            rxn.upper_bound = 1000
    
    for rxn in comp_model.reactions:
        if rxn.id in media:
            rxn.lower_bound = -media[rxn.id]  # set media uptake rates


    # for reaction in comp_model.reactions:
    #     print(reaction.id, reaction.reaction, reaction.lower_bound, reaction.upper_bound)

    solution = comp_model.optimize()
    # cb.io.save_json_model(comp_model, f"cFBA_Models/cFBA_sim{sit_idx}.json") # save for others to avoid re-building
    print(f"{comp_model.id}, Objective value: {solution.objective_value}")
    print(solution.fluxes)

      # store results
    cfba_results = pd.DataFrame(columns=["First_Org_Flux", "Second_Org_Max_Flux", "Second_Org_Min_Flux", "3rd_Org_Max_Flux", "4th_Org_Max_Flux"])

    if sit_idx != 2:
        # ========= Set community growth constraint =============
        comm_growth = solution.fluxes[[rxn.id for rxn in objective_reactions]].sum()
        community_expr = sum(comp_model.reactions.get_by_id(f"DM_biomass(e)__Org{i+1}").flux_expression for i in range(len(models)))
        # Add constraint to model
        constraint = optlang.Constraint(
            community_expr,
            lb=comm_growth,  # lower bound
            ub=comm_growth,  # upper bound
            name="community_growth_constraint"
            )
        comp_model.solver.add(constraint)



        # ======= Get Min/Max Flux for organism 1 =============
        # minimize flux of first organism
        comp_model.objective = comp_model.reactions.get_by_id("DM_biomass(e)__Org1")
        comp_model.objective.direction = 'min'
        solution_min_s1 = comp_model.optimize()

        # maximize flux of first organism
        comp_model.objective.direction = 'max'
        solution_max_s1 = comp_model.optimize()

        

      
        print("Min and Max flux for first organism:", solution_min_s1.objective_value, solution_max_s1.objective_value)
        

        # ======== Vary first organism flux between min and max and get second organism min/max ========
        for val in np.linspace(solution_min_s1.objective_value, solution_max_s1.objective_value, 101):
            # update constraint on first organism
            comp_model.reactions.get_by_id("DM_biomass(e)__Org1").upper_bound = val
            comp_model.reactions.get_by_id("DM_biomass(e)__Org1").lower_bound = val

            # optimize for second organism
            comp_model.objective = comp_model.reactions.get_by_id("DM_biomass(e)__Org2")
            comp_model.objective.direction = 'max'
            solution_max_s2 = comp_model.optimize()
            # minimize then maximize org 2
            comp_model.objective.direction = 'min'
            solution_min_s2 = comp_model.optimize()

            # store results
            cfba_results = pd.concat([cfba_results, pd.DataFrame({
                "First_Org_Flux": val,
                "Second_Org_Max_Flux": solution_max_s2.objective_value,
                "Second_Org_Min_Flux": solution_min_s2.objective_value,
                "3rd_Org_Max_Flux": 0,
                "4th_Org_Max_Flux": 0
            }, index=[0])], ignore_index=True)

    if sit_idx == 2:    
        cfba_results = pd.concat([cfba_results, pd.DataFrame({
                "First_Org_Flux": solution["DM_biomass(e)__Org1"],
                "Second_Org_Max_Flux": solution["DM_biomass(e)__Org2"],
                "Second_Org_Min_Flux": 0,
                "3rd_Org_Max_Flux": solution["DM_biomass(e)__Org3"],
                "4th_Org_Max_Flux": solution["DM_biomass(e)__Org4"]
            }, index=[0])], ignore_index=True)
        
    cfba_results.to_csv(f"Results/cFBA/cFBA_results_{names[sit_idx]}.csv", index=False)




    

Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p


Output()

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.

Using regular expression found the following compartments:c, e, p

DM_biomass(e)__Org1 biomass[c]__Org1 <=>  -1000.0 1000.0
pbiosynthesis__Org1  --> proteinsynth[c]__Org1 0.0 1000.0
biomass525__Org1 0.0078094 ACP[c]__Org1 + 0.092476 PGP[c]__Org1 + 0.50006 ala_L[c]__Org1 + 0.0078094 amet[c]__Org1 + 0.28827 arg_L[c]__Org1 + 0.23468 asn_L[c]__Org1 + 0.23468 asp_L[c]__Org1 + 40.1701 atp[c]__Org1 + 0.0078094 ca2[c]__Org1 + 0.0078094 cl[c]__Org1 + 0.0078094 coa[c]__Org1 + 0.0078094 cobalt2[c]__Org1 + 0.025039 colipa[c]__Org1 + 0.12988 ctp[c]__Org1 + 0.0078094 cu2[c]__Org1 + 0.08898 cys_L[c]__Org1 + 0.011747 datp[c]__Org1 + 0.011747 dctp[c]__Org1 + 0.011747 dgtp[c]__Org1 + dnarep[c]__Org1 + 0.02504 dtdprmn[c]__Org1 + 0.011747 dttp[c]__Org1 + 0.0078094 fad[c]__Org1 + 0.0078094 fe2[c]__Org1 + 0.0078094 fe3[c]__Org1 + 0.25601 gln_L[c]__Org1 + 0.25601 glu_L[c]__Org1 + 0.5958 gly[c]__Org1 + 0.2091 gtp[c]__Org1 + 34.7965 h2o[c]__Org1 + 0.092623 his_L[c]__Org1 + 0.28255 ile_L[c]__Org1 + 0.0078094 k[c]__Org1 + 0.02504 kdo2lipid4L[c]__Org1 + 0.43866 leu_L[c]__Org1 + 

/home/rseag/anaconda3/envs/M3/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [1]:
import pandas as pd
import cobra as cb
import gifba 
from micom import Community
import numpy as np
import optlang

micom_results = pd.DataFrame(columns=["simulation", "tradeoff", "Org1_growth", "Org2_growth", ])

model_paths = [
        ["AGORA2_Models/Escherichia_coli_str_K_12_substr_MG1655.mat",
        "AGORA2_Models/Bacteroides_thetaiotaomicron_3731.mat"],
        ['AGORA2_Models/Bifidobacterium_longum_infantis_ATCC_15697.mat',
         'AGORA2_Models/Eubacterium_hallii_DSM_3353.mat'],
         ["AGORA2_Models/Clostridium_difficile_R20291.mat",
          "AGORA2_Models/Clostridium_hiranonis_TO_931_DSM_13275.mat",
          "AGORA2_Models/Bacteroides_thetaiotaomicron_3731.mat",
          "AGORA2_Models/Eggerthella_lenta_DSM_11767.mat"]
]
media_list = []

# EC/BT
#example glucose minimal media
min_med_ids_ex = ['EX_glc_D(e)','EX_so4(e)','EX_nh4(e)','EX_pi(e)','EX_cys_L(e)',
			'EX_mn2(e)','EX_cl(e)','EX_ca2(e)','EX_mg2(e)','EX_cu2(e)',
			'EX_cobalt2(e)','EX_fe2(e)','EX_fe3(e)','EX_zn2(e)','EX_k(e)']
# Define medium uptake flux bounds
min_med_fluxes_ex = [-10,-100,-100,-100,-100,
					-100,-100,-100,-100,-100,-100,-100,-100,-100,-100]
media = dict(zip(min_med_ids_ex, min_med_fluxes_ex))
media_list.append(media)

# BI/AH
media = {
	'EX_o2(e)': 0, #aerobic/anaerobic
	'EX_h2o(e)': -1000,
	'EX_pi(e)': -1000,
	'EX_fe2(e)': -1000,
	'EX_fe3(e)': -1000,
	'EX_zn2(e)': -1000,
	'EX_so4(e)': -1000,
	'EX_cu2(e)': -1000,
	'EX_k(e)': -1000,
	'EX_mg2(e)': -1000,
	'EX_mn2(e)': -1000,
	'EX_cd2(e)': -1000,
	'EX_cl(e)': -1000,
	'EX_ca2(e)': -1000,
	'EX_cobalt2(e)': -1000,
	'EX_glc_D(e)': -10,
	'EX_nh4(e)': -20,

	'EX_ribflv(e)': -1000,
	'EX_pnto_R(e)': -1000,
	'EX_nac(e)': -1000,
	'EX_his_L(e)': -1000,
	'EX_asn_L(e)': -1000,
	'EX_glycys(e)': -1000,

	'EX_lys_L(e)': -1000,
	'EX_ala_L(e)': -1000,
	'EX_met_L(e)': -1000,
	'EX_leu_L(e)': -1000,
	'EX_hxan(e)': -1000,    
    'EX_glyglu(e)': -1000
	# 'EX_ser_L(e)': -1 #-1
}
media_list.append(media)


# CDiff community
euro_media = "AGORA2_Models/euro_diet.tsv"
euro_media = pd.read_csv(euro_media, sep="\t", header=0, index_col=0).to_dict()
euro_media = euro_media["Flux Value"]
euro_media = {ex.replace("[e]", "(e)"): -flux for ex, flux in euro_media.items()}
euro_media["EX_cobalt2(e)"] = -1000 #cobalt
euro_media["EX_so4(e)"] = -1000 # sulfate
euro_media["EX_adn(e)"] = -1000 #Adenosine
euro_media["EX_cytd(e)"] = -1000 # cytidine
euro_media["EX_thymd(e)"] = -1000 # thymidine
euro_media["EX_hspg(e)"] = -1 # Heparan Sulfate Proteoglycan
euro_media["EX_26dap_M(e)"] = -10 # Meso-2,6-Diaminoheptanedioate
euro_media["EX_cgly(e)"] = -1000 # Cysteinylglycine - dipep
euro_media["EX_glyasn(e)"] = -1000 # Glycylasparagine - dipep
euro_media["EX_nmn(e)"] = -1000 # Nicotinamide ribotide
euro_media["EX_pheme(e)"] = -1000 #p heme
euro_media["EX_sheme(e)"] = -1000 # s heme
euro_media["EX_spmd(e)"] = -1000 # spermidine
media_list.append(euro_media)

names = ["EC_BT", "BI_AH", "Cdiff_Community"]



for sit_idx in range(3): # EC/BT, BI/AH, Cdiff Comm
    # get media from giFBA
    media = media_list[sit_idx]
    models = [cb.io.load_matlab_model(path) for path in model_paths[sit_idx]]
    if sit_idx == 2:
        community = gifba.gifbaObject(models, [media, 0.1], # minimal media
									rel_abund=[0.1, 0.3, 0.3, 0.3]) 
        media = community.media
        
    id2index = {model.id: idx for idx, model in enumerate(models)}
    media = {k.replace("(e)", "_m"): -v for k, v in media.items()}
    rxn_up_bounds = {model_idx: {rxn.id : rxn.upper_bound for rxn in models[model_idx].reactions} for model_idx in range(len(models))}
    rxn_low_bounds = {model_idx: {rxn.id : rxn.lower_bound for rxn in models[model_idx].reactions} for model_idx in range(len(models))}

    # print(rxn_up_bounds)

    # make sure 2b (same model, different abundances) uses correct abundances
    # abund = [0.2, 0.8] if sit_idx == "2b" else [0.5, 0.5]
    # sim_idx = sit_idx if sit_idx != "2b" else "2a"
    if sit_idx !=2:
        abund = [0.5, 0.5]
        ids = [f"Org{i+1}" for i in range(len(models))]
    else:
        abund = [0.1, 0.3, 0.3, 0.3]
        ids = [f"Org{i+1}" for i in range(len(models))]

    # create community dataframe
    community = pd.DataFrame({
        "id": ids,
        "file": model_paths[sit_idx],
        "abundance": abund
    })

    # create micom community
    community = Community(community)

    comp_model = cb.Model(f"compartmentalized_model_{sit_idx}")

    tmp = cb.Model("tmp")
    tmp.add_reactions([r.copy() for r in community.reactions])  # use community.model

    for rxn in tmp.reactions:
        # new_stoich = {}
        # for met, coef in list(rxn.metabolites.items()):  # snapshot (met->coef)
        #     if met.compartment == "m" and len(rxn.metabolites) == 1:
        #         new_stoich[met] = -1.0  # or coef * something, up to you
        #     elif met.compartment == "m":
        #         new_stoich[met] = 1.0  # or coef * something, up to you
        #     else:
        #         # WARNING: your compartments are like 'c__Org1', 'e__Org2' etc
        #         # met.compartment[-1] works only if last char is '1'/'2'
        #         model_num = int(met.compartment[-1]) - 1
        #         new_stoich[met] = coef / abund[model_num]
        
        # # overwrite stoichiometry
        # rxn.add_metabolites(new_stoich, combine=False)
        if "_m" not in rxn.id and len(rxn.metabolites) != 1:
            model_num = int(rxn.id.split("__")[-1][-1]) - 1
            orig_id = rxn.id.replace("__Org"+str(model_num+1), "")

            if "EX_" not in orig_id:
                rxn.lower_bound = rxn_low_bounds[model_num][orig_id]# * abund[model_num]
                rxn.upper_bound = rxn_up_bounds[model_num][orig_id]# * abund[model_num]


    comp_model.add_reactions([r.copy() for r in tmp.reactions])

    
    for reaction in comp_model.reactions:
        if "biomass" in reaction.id.lower() or "bio" in reaction.id.lower():
            print(reaction.id, reaction.reaction, reaction.lower_bound, reaction.upper_bound)
    
    # change objective to community growth (weighted sum of biomass reactions)
    objective_reactions = [rxn for rxn in comp_model.reactions if "dm_biomass(e)" in rxn.id.lower()]
    objective_rxns_coef = abund #[1 for _ in range(len(objective_reactions))]
    comp_model.objective = dict(zip(objective_reactions, objective_rxns_coef))
    comp_model.objective.direction = "max"

    for rxn in comp_model.reactions:
        if len(rxn.metabolites) == 1 and list(rxn.metabolites.keys())[0].compartment == "m" and "biomass" not in rxn.id.lower():
        # if rxn.boundary and "_m" in rxn.id:
            # name = rxn.id.split("_m")[0]
            rxn.lower_bound = 0
            rxn.upper_bound = 1000
    
    for rxn in comp_model.reactions:
        if rxn.id in media:
            rxn.lower_bound = -media[rxn.id]  # set media uptake rates


    # for reaction in comp_model.reactions:
    #     print(reaction.id, reaction.reaction, reaction.lower_bound, reaction.upper_bound)

    solution = comp_model.optimize()
    # cb.io.save_json_model(comp_model, f"cFBA_Models/cFBA_sim{sit_idx}.json") # save for others to avoid re-building
    print(f"{comp_model.id}, Objective value: {solution.objective_value}")
    print([abund[idx] * solution[rxn.id] for idx, rxn in enumerate(objective_reactions)])
    print(solution.fluxes)

      # store results
    cfba_results = pd.DataFrame(columns=["First_Org_Flux", "Second_Org_Max_Flux", "Second_Org_Min_Flux", "3rd_Org_Max_Flux", "4th_Org_Max_Flux"])

    if sit_idx != 2:
        # ========= Set community growth constraint =============
        comm_growth = solution.fluxes[[rxn.id for rxn in objective_reactions]]@np.array(abund)
        community_expr = sum(abund[i] * comp_model.reactions.get_by_id(f"DM_biomass(e)__Org{i+1}").flux_expression for i in range(len(models)))
        # Add constraint to model
        constraint = optlang.Constraint(
            community_expr,
            lb=comm_growth,  # lower bound
            ub=comm_growth,  # upper bound
            name="community_growth_constraint"
            )
        comp_model.solver.add(constraint)



        # ======= Get Min/Max Flux for organism 1 =============
        # minimize flux of first organism
        comp_model.objective = comp_model.reactions.get_by_id("DM_biomass(e)__Org1")
        comp_model.objective.direction = 'min'
        solution_min_s1 = comp_model.optimize()

        # maximize flux of first organism
        comp_model.objective.direction = 'max'
        solution_max_s1 = comp_model.optimize()

        

        print(comm_growth)
        print("Min and Max flux for first organism:", 0.5* solution_min_s1.objective_value, 0.5*solution_max_s1.objective_value)
        

        # ======== Vary first organism flux between min and max and get second organism min/max ========
        for val in np.linspace(solution_min_s1.objective_value, solution_max_s1.objective_value, 101):
            # update constraint on first organism
            comp_model.reactions.get_by_id("DM_biomass(e)__Org1").upper_bound = val
            comp_model.reactions.get_by_id("DM_biomass(e)__Org1").lower_bound = val

            # optimize for second organism
            comp_model.objective = comp_model.reactions.get_by_id("DM_biomass(e)__Org2")
            comp_model.objective.direction = 'max'
            solution_max_s2 = comp_model.optimize()
            # minimize then maximize org 2
            comp_model.objective.direction = 'min'
            solution_min_s2 = comp_model.optimize()

            # store results
            cfba_results = pd.concat([cfba_results, pd.DataFrame({
                "First_Org_Flux": abund[0] * val,
                "Second_Org_Max_Flux": abund[1] * solution_max_s2.objective_value,
                "Second_Org_Min_Flux": abund[1] * solution_min_s2.objective_value,
                "3rd_Org_Max_Flux": 0,
                "4th_Org_Max_Flux": 0
            }, index=[0])], ignore_index=True)

    if sit_idx == 2:    
        cfba_results = pd.concat([cfba_results, pd.DataFrame({
                "First_Org_Flux": abund[0] * solution["DM_biomass(e)__Org1"],
                "Second_Org_Max_Flux": abund[1] * solution["DM_biomass(e)__Org2"],
                "Second_Org_Min_Flux": 0,
                "3rd_Org_Max_Flux": abund[2] * solution["DM_biomass(e)__Org3"],
                "4th_Org_Max_Flux": abund[3] * solution["DM_biomass(e)__Org4"]
            }, index=[0])], ignore_index=True)
        
    cfba_results.to_csv(f"Results/cFBA/cFBA_results_{names[sit_idx]}.csv", index=False)




    

Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p


Output()

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.

Using regular expression found the following compartments:c, e, p

...

Using regular expression found the following compartments:c, e, p

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.

Using regular expression found the following compartments:c, e

DM_biomass(e)__Org1 biomass[c]__Org1 <=>  -1000.0 1000.0
pbiosynthesis__Org1  --> proteinsynth[c]__Org1 0.0 1000.0
biomass205__Org1 0.0079397 10fthf[c]__Org1 + 0.0079397 2dmmq8[c]__Org1 + 0.0079397 5mthf[c]__Org1 + 0.0079397 ACP[c]__Org1 + 0.0018061 PGP[c]__Org1 + 0.0079397 adocbl[c]__Org1 + 0.0018061 ai17tca1[c]__Org1 + 0.0018061 ai17tcaacgam[c]__Org1 + 0.0018061 ai17tcaala_D[c]__Org1 + 0.0018061 ai17tcaglc[c]__Org1 + 0.26756 ala_L[c]__Org1 + 0.0079397 amet[c]__Org1 + 0.1934 arg_L[c]__Org1 + 0.14833 asn_L[c]__Org1 + 0.14833 asp_L[c]__Org1 + 41.2914 atp[c]__Org1 + 0.0079397 ca2[c]__Org1 + 0.0079397 cl[c]__Org1 + 0.0085871 clpn180[c]__Org1 + 0.0085871 clpnai17[c]__Org1 + 0.0085871 clpni17[c]__Org1 + 0.0079397 coa[c]__Org1 + 0.0079397 cobalt2[c]__Org1 + 0.026124 ctp[c]__Org1 + 0.0079397 cu2[c]__Org1 + 0.056954 cys_L[c]__Org1 + 0.013437 datp[c]__Org1 + 0.013437 dctp[c]__Org1 + 0.013437 dgtp[c]__Org1 + dnarep[c]__Org1 + 0.013437 dttp[c]__Org1 + 0.0079397 fad[c]__Org1 + 0.0079397 fe2[c]__Or

/tmp/ipykernel_126777/776392950.py:252: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cfba_results = pd.concat([cfba_results, pd.DataFrame({
